In [49]:
import os
import polars as pl
from typing import List, Dict, Tuple 

from llm_benchmark.utils.dataset import Dataset
from llm_benchmark.utils.benchmark import seshat_setup
from llm_benchmark.data import eval

def tally_single_model(dataset: Dataset,
                       model_filename: str,
                       model: str
                       ) -> pl.DataFrame:
    tally_df: pl.DataFrame = eval.aggregate_entry_per_hierarchy(
        dataset=dataset, 
        dir=os.path.join("/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/evaluation/", 
                         model_filename, 
                         "wf_answers"),
        model=model
        )
    return tally_df
to_tally: Dict[str, str] = {
    "22_04_2026/run_3_Qwen_Qwen-7B-Chat": "Qwen-7B-Chat",
    "14_05_2026/run_9_Qwen_Qwen3-8B": "Qwen3-8B-Chat",
    "14_05_2026/run_11_Qwen_Qwen2.5-7B": "Qwen2.5-7B-Chat",
}

dataset: Dataset = seshat_setup(
    seshat_cache_dir="/Users/apple/Documents/github/neurips_llms/llm-bechmark/db/seshat", 
    force=False
    )

dfs: List[pl.DataFrame] = []

for model_filename, model in to_tally.items():
    dfs.append(tally_single_model(
        dataset=dataset, 
        model_filename=model_filename, 
        model=model
        ))
df: pl.DataFrame = pl.concat(dfs)\
    .with_columns(
        (pl.col("model_answer") == pl.col("actual_answer"))
        .cast(pl.Int8)
        .fill_null(0)
        .alias("result")
    )


Attempting dataset refresh. False
core/macro-regions https://seshat-db.com/api/core/macro-regions/
Ignoring polity core/macro-regions as per configuration.
core/regions https://seshat-db.com/api/core/regions/
Ignoring polity core/regions as per configuration.
core/ngas https://seshat-db.com/api/core/ngas/
Ignoring polity core/ngas as per configuration.
core/polities https://seshat-db.com/api/core/polities/
Ignoring polity core/polities as per configuration.
core/capitals https://seshat-db.com/api/core/capitals/
Ignoring polity core/capitals as per configuration.
core/nga-polity-relations https://seshat-db.com/api/core/nga-polity-relations/
Ignoring polity core/nga-polity-relations as per configuration.
core/sections https://seshat-db.com/api/core/sections/
Ignoring polity core/sections as per configuration.
core/subsections https://seshat-db.com/api/core/subsections/
Ignoring polity core/subsections as per configuration.
core/variable-hierarchies https://seshat-db.com/api/core/variable

Qwen2.5-7B-Chat: 100%|██████████| 49/49 [00:19<00:00,  2.57it/s]


In [35]:
print(df.columns)

['seshat_entry_id', 'question_entry_id', 'model_answer', 'actual_answer', 'region_idx', 'region_str', 'macro_idx', 'macro_str', 'section_idx', 'section_str', 'subsection_idx', 'subsection_str', 'year_range', 'endpoint', 'llm_model', 'result']


In [36]:
print(df)

shape: (52_608, 16)
┌────────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬────────┐
│ seshat_ent ┆ question_e ┆ model_ans ┆ actual_an ┆ … ┆ year_rang ┆ endpoint  ┆ llm_model ┆ result │
│ ry_id      ┆ ntry_id    ┆ wer       ┆ swer      ┆   ┆ e         ┆ ---       ┆ ---       ┆ ---    │
│ ---        ┆ ---        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ str       ┆ str       ┆ i8     │
│ i64        ┆ i64        ┆ str       ┆ str       ┆   ┆ str       ┆           ┆           ┆        │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪════════╡
│ 0          ┆ 0          ┆ i'm       ┆ present   ┆ … ┆ 1500 CE - ┆ wf/limb-p ┆ Qwen-7B-C ┆ 0      │
│            ┆            ┆           ┆           ┆   ┆ 2000 CE   ┆ rotection ┆ hat       ┆        │
│            ┆            ┆           ┆           ┆   ┆           ┆ s_answers ┆           ┆        │
│ 1          ┆ 1          ┆ absent    ┆ present   ┆ … ┆ 1000 CE - ┆ wf/

In [37]:
def group(
        df: pl.DataFrame,
        metric: str,
        ) -> pl.DataFrame:
    return df\
    .select(["llm_model", "endpoint", "result", metric])\
    .group_by(["llm_model", "endpoint", metric])\
    .agg(
        pl.col("result").mean().alias("per_metric")
    )\
    .group_by(["llm_model", metric])\
    .agg(
        pl.format(
            "{} [{}, {}]",
            (pl.col("per_metric").mean() * 100).round(1),
            (pl.col("per_metric").min() * 100).round(1),
            (pl.col("per_metric").max() * 100).round(1)
            ).alias("metrics")
    )\
    .pivot(
        on="llm_model",
        index=metric,
        values="metrics"
    )\
    .sort([metric])

In [38]:
metric = "macro_str"
with pl.Config(set_tbl_rows=500):
    print(
        df\
        .select(["llm_model", "endpoint", "result", metric])\
        .group_by(["llm_model", "endpoint", metric])\
        .agg(
            pl.col("result").mean().alias("per_metric")
        )\
        .group_by(["llm_model", metric]).head(500)
    )

shape: (1_470, 4)
┌─────────────────┬──────────────────────────────┬─────────────────────────────────┬────────────┐
│ llm_model       ┆ macro_str                    ┆ endpoint                        ┆ per_metric │
│ ---             ┆ ---                          ┆ ---                             ┆ ---        │
│ str             ┆ str                          ┆ str                             ┆ f64        │
╞═════════════════╪══════════════════════════════╪═════════════════════════════════╪════════════╡
│ Qwen2.5-7B-Chat ┆ South Asia                   ┆ wf/scaled-armors_answers        ┆ 0.0        │
│ Qwen2.5-7B-Chat ┆ South Asia                   ┆ wf/sling-siege-engines_answers  ┆ 0.0        │
│ Qwen2.5-7B-Chat ┆ South Asia                   ┆ wf/bronzes_answers              ┆ 0.0        │
│ Qwen2.5-7B-Chat ┆ South Asia                   ┆ wf/swords_answers               ┆ 0.0        │
│ Qwen2.5-7B-Chat ┆ South Asia                   ┆ wf/small-vessel-canoe-etc_answ… ┆ 0.0        │
│ 

In [39]:
# Filter down to Qwen-7B-Chat where the endpoint score is exactly 0
zero_endpoints = (
    df
    .filter(pl.col("llm_model") == "Qwen-7B-Chat")
    .group_by(["endpoint", metric])
    .agg(pl.col("result").mean().alias("per_metric"))
    .filter(pl.col("per_metric") == 0)
)

print(zero_endpoints)

shape: (24, 3)
┌─────────────────────────────────┬──────────────────────────────┬────────────┐
│ endpoint                        ┆ macro_str                    ┆ per_metric │
│ ---                             ┆ ---                          ┆ ---        │
│ str                             ┆ str                          ┆ f64        │
╞═════════════════════════════════╪══════════════════════════════╪════════════╡
│ wf/helmets_answers              ┆ Oceania-Australia            ┆ 0.0        │
│ wf/long-walls_answers           ┆ Central and Northern Eurasia ┆ 0.0        │
│ wf/long-walls_answers           ┆ South America and Caribbean  ┆ 0.0        │
│ wf/long-walls_answers           ┆ Europe                       ┆ 0.0        │
│ wf/tension-siege-engines_answe… ┆ South America and Caribbean  ┆ 0.0        │
│ wf/leathers_answers             ┆ South America and Caribbean  ┆ 0.0        │
│ wf/daggers_answers              ┆ Oceania-Australia            ┆ 0.0        │
│ wf/spears_answers      

In [15]:
print(df.columns)
print(df['section_str'].unique().sort())

['seshat_entry_id', 'question_entry_id', 'model_answer', 'actual_answer', 'region_idx', 'region_str', 'macro_idx', 'macro_str', 'section_idx', 'section_str', 'subsection_idx', 'subsection_str', 'year_range', 'endpoint', 'llm_model', 'result']
shape: (7,)
Series: 'section_str' [str]
[
	"Animals used in warfare"
	"Armor"
	"Fortifications"
	"Handheld weapons"
	"Military use of Metals"
	"Naval technology"
	"Projectiles"
]


In [41]:
macro_region: pl.DataFrame = group(df=df, metric="macro_str").sort(['macro_str'])

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(macro_region.head(500))

shape: (10, 4)
┌──────────────────────────────┬────────────────┬─────────────────┬──────────────────┐
│ macro_str                    ┆ Qwen3-8B-Chat  ┆ Qwen2.5-7B-Chat ┆ Qwen-7B-Chat     │
│ ---                          ┆ ---            ┆ ---             ┆ ---              │
│ str                          ┆ str            ┆ str             ┆ str              │
╞══════════════════════════════╪════════════════╪═════════════════╪══════════════════╡
│ Africa                       ┆ 0.0 [0.0, 0.0] ┆ 0.1 [0.0, 2.5]  ┆ 29.9 [0.0, 65.8] │
│ Central and Northern Eurasia ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 22.5 [0.0, 61.1] │
│ East Asia                    ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 32.4 [0.0, 65.0] │
│ Europe                       ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 29.0 [0.0, 56.4] │
│ North America                ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 28.7 [0.0, 66.7] │
│ Oceania-Australia            ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 27.1 [0.0, 87.5] │
│ South America and Caribbea

In [48]:
region: pl.DataFrame = group(df=df, metric="region_str").sort(['region_str'])

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(region.head(500))

shape: (34, 4)
┌─────────────────────────┬────────────────┬─────────────────┬───────────────────┐
│ region_str              ┆ Qwen3-8B-Chat  ┆ Qwen2.5-7B-Chat ┆ Qwen-7B-Chat      │
│ ---                     ┆ ---            ┆ ---             ┆ ---               │
│ str                     ┆ str            ┆ str             ┆ str               │
╞═════════════════════════╪════════════════╪═════════════════╪═══════════════════╡
│ Afghanistan             ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 26.6 [0.0, 85.7]  │
│ Amazonia                ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 23.1 [0.0, 100.0] │
│ Anatolia-Caucasus       ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 28.3 [0.0, 51.9]  │
│ Andes                   ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 18.9 [0.0, 71.4]  │
│ Arabia                  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 16.2 [0.0, 71.4]  │
│ Caribbean               ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 20.4 [0.0, 100.0] │
│ East Africa             ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 16.7 [0.0

In [19]:
sections: pl.DataFrame = group(df=df, metric="section_str").sort(['section_str'])

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(sections.head(500))

shape: (7, 4)
┌─────────────────────────┬────────────────┬─────────────────┬───────────────────┐
│ section_str             ┆ Qwen3-8B-Chat  ┆ Qwen2.5-7B-Chat ┆ Qwen-7B-Chat      │
│ ---                     ┆ ---            ┆ ---             ┆ ---               │
│ str                     ┆ str            ┆ str             ┆ str               │
╞═════════════════════════╪════════════════╪═════════════════╪═══════════════════╡
│ Animals used in warfare ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 30.9 [14.7, 42.4] │
│ Armor                   ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 23.4 [18.4, 35.3] │
│ Fortifications          ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 23.5 [0.0, 48.6]  │
│ Handheld weapons        ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.3]  ┆ 25.6 [18.5, 37.6] │
│ Military use of Metals  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 31.4 [28.3, 34.4] │
│ Naval technology        ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 25.9 [23.4, 29.0] │
│ Projectiles             ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]  ┆ 32.4 [19.3

In [6]:
subsections: pl.DataFrame = group(df=df, metric="subsection_str").sort(['subsection_str'])

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(subsections.head(500))

shape: (1, 4)
┌────────────────┬──────────────┬─────────────────┬───────────────┐
│ subsection_str ┆ Qwen-7B-Chat ┆ Qwen2.5-7B-Chat ┆ Qwen3-8B-Chat │
│ ---            ┆ ---          ┆ ---             ┆ ---           │
│ null           ┆ f64          ┆ f64             ┆ f64           │
╞════════════════╪══════════════╪═════════════════╪═══════════════╡
│ null           ┆ 27.3         ┆ 0.0             ┆ 0.0           │
└────────────────┴──────────────┴─────────────────┴───────────────┘


In [20]:
subsections: pl.DataFrame = group(df=df, metric="year_range").sort(['year_range'])

with pl.Config(tbl_rows=200, set_tbl_width_chars=100):
    print(subsections.head(500))

shape: (56, 4)
┌──────────────────────┬───────────────────┬────────────────┬──────────────────┐
│ year_range           ┆ Qwen-7B-Chat      ┆ Qwen3-8B-Chat  ┆ Qwen2.5-7B-Chat  │
│ ---                  ┆ ---               ┆ ---            ┆ ---              │
│ str                  ┆ str               ┆ str            ┆ str              │
╞══════════════════════╪═══════════════════╪════════════════╪══════════════════╡
│ 0 CE - 0 CE          ┆ 27.1 [0.0, 80.0]  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 0 CE - 1000 CE       ┆ 18.4 [0.0, 100.0] ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 0 CE - 500 CE        ┆ 26.7 [0.0, 66.7]  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 1000 BCE - 1000 BCE  ┆ 27.1 [0.0, 77.8]  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 1000 BCE - 500 BCE   ┆ 25.3 [0.0, 63.2]  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 1000 CE - 1000 CE    ┆ 28.8 [0.0, 57.9]  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 1000 CE - 1500 CE    ┆ 24.7 [0.0, 58.8]  ┆ 0.0 [0.0, 0.0] ┆ 0.0 [0.0, 0.0]   │
│ 13500 BCE -